In [2]:
# setup: load config and normalized AnnData
import os
os.chdir("C:/Users/ASRENOVIN/Desktop/spatial-foundation-graph-transformer")

import yaml
import numpy as np
import anndata as ad

with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

adata = ad.read_h5ad("data/processed/adata_normalized.h5ad")
print(adata)
print("\nHVGs:", adata.var["highly_variable"].sum())

AnnData object with n_obs × n_vars = 3661 × 20955
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'log1p', 'pca', 'spatial'
    obsm: 'X_pca', 'spatial'
    varm: 'PCs'

HVGs: 3000


In [3]:
# write_scgpt_provider: implement scGPT embedding provider with full API calls
scgpt_code = '''from __future__ import annotations
import logging
from pathlib import Path
from typing import Any

import numpy as np
import anndata as ad

from embeddings.base import EmbeddingProvider

logger = logging.getLogger(__name__)


class scGPTEmbeddingProvider(EmbeddingProvider):
    """
    Embedding provider using pretrained scGPT (Chen et al. 2023).

    scGPT is a foundation model pretrained on 33 million human single cells.
    It encodes each cell/spot into a biologically meaningful latent space
    by treating genes as tokens and learning gene-gene relationships via
    a transformer architecture.

    Reference:
        Chen et al. (2023). scGPT: toward building a foundation model for
        single-cell multi-omics using generative AI. Nature Methods.
        https://doi.org/10.1038/s41592-024-02201-0

    Requirements:
        - pip install scgpt
        - Pretrained weights downloaded to cfg["embeddings"]["scgpt"]["model_dir"]
        - GPU strongly recommended (CPU inference is very slow for 3000+ spots)
        - Linux recommended (Windows has torchtext binary compatibility issues)

    Weights download:
        import scgpt
        scgpt.download_pretrained_model("human", save_dir="data/raw/scgpt_weights")

    Usage:
        Set embeddings.provider: "scgpt" in configs/default.yaml
        Then run: provider = get_embedding_provider(cfg); adata = provider.embed(adata)
    """

    def __init__(self, cfg: dict[str, Any]) -> None:
        super().__init__(cfg)
        self.model_dir  = Path(cfg["embeddings"]["scgpt"]["model_dir"])
        self.gene_col   = cfg["embeddings"]["scgpt"].get("gene_col", "gene_name")
        self.batch_size = cfg["embeddings"]["scgpt"].get("batch_size", 32)
        self.device     = cfg["embeddings"]["scgpt"].get("device", "cuda")

    def embed(self, adata: ad.AnnData) -> ad.AnnData:
        """
        Generate scGPT embeddings for each spot in adata.

        Steps:
            1. Subset to highly variable genes
            2. Tokenize gene expression with scGPT gene vocabulary
            3. Run transformer encoder in batches
            4. Store embeddings in adata.obsm["X_embedding"]

        Parameters
        ----------
        adata : AnnData
            Normalized AnnData object (log1p counts, HVGs flagged).

        Returns
        -------
        AnnData
            Same object with adata.obsm["X_embedding"] populated (shape: n_spots x 512).
        """
        try:
            import scgpt
            from scgpt.tasks import embed_data
        except ImportError as e:
            raise ImportError(
                "scgpt is not installed or failed to load.\\n"
                "Install with: pip install scgpt\\n"
                "Note: requires Linux + GPU for full inference.\\n"
                f"Original error: {e}"
            )

        if not self.model_dir.exists():
            raise FileNotFoundError(
                f"scGPT model weights not found at {self.model_dir}.\\n"
                "Download with:\\n"
                "  import scgpt\\n"
                "  scgpt.download_pretrained_model(\\'human\\', "
                "save_dir=\\'data/raw/scgpt_weights\\')"
            )

        logger.info("Running scGPT embedding on %d spots...", adata.n_obs)
        logger.info("Model dir : %s", self.model_dir)
        logger.info("Device    : %s", self.device)
        logger.info("Batch size: %d", self.batch_size)

        # subset to HVGs for efficiency
        adata_hvg = adata[:, adata.var["highly_variable"]].copy()
        logger.info("Using %d HVGs as input tokens", adata_hvg.n_vars)

        # scGPT embed_data expects:
        #   adata with raw-ish counts in .X
        #   gene names as var_names (HGNC symbols)
        #   gene_col key in adata.var pointing to gene name column
        embeddings = embed_data(
            adata_hvg,
            model_dir=str(self.model_dir),
            gene_col="index",          # var_names are already HGNC symbols
            batch_size=self.batch_size,
            device=self.device,
            use_fast_transformer=False, # flash_attn not required
            return_new_adata=False,
        )

        # embed_data stores result in adata_hvg.obsm["X_scGPT"]
        X_scgpt = adata_hvg.obsm["X_scGPT"]
        adata.obsm["X_scGPT"]    = X_scgpt
        adata.obsm["X_embedding"] = X_scgpt.copy()

        np.save("data/embeddings/X_scgpt.npy", X_scgpt)
        logger.info("scGPT embedding shape: %s", X_scgpt.shape)
        print(f"scGPT embedding shape: {X_scgpt.shape}")

        return adata
'''

with open("src/embeddings/scgpt_provider.py", "w", encoding="utf-8") as f:
    f.write(scgpt_code)

print("Created: src/embeddings/scgpt_provider.py")

Created: src/embeddings/scgpt_provider.py


In [5]:
# register_scgpt_in_factory: update factory.py to include scGPT provider
factory_code = '''from __future__ import annotations
from typing import Any
import anndata as ad
from embeddings.base import EmbeddingProvider
from embeddings.pca_provider import PCAEmbeddingProvider


def get_embedding_provider(cfg: dict[str, Any]) -> EmbeddingProvider:
    """
    Return the correct EmbeddingProvider based on cfg["embeddings"]["provider"].

    Supported providers:
        "pca"      - PCA on precomputed X_pca (default, no extra dependencies)
        "scgpt"    - scGPT foundation model (requires GPU + Linux recommended)
        "geneformer" - Geneformer foundation model (not yet implemented)
        "uce"      - UCE foundation model (not yet implemented)
    """
    provider = cfg["embeddings"]["provider"]

    if provider == "pca":
        return PCAEmbeddingProvider(cfg)

    if provider == "scgpt":
        from embeddings.scgpt_provider import scGPTEmbeddingProvider
        return scGPTEmbeddingProvider(cfg)

    if provider in ("geneformer", "uce"):
        raise NotImplementedError(
            f"Embedding provider \'{provider}\' is not yet implemented.\\n"
            f"To add it: create src/embeddings/{provider}_provider.py "
            f"subclassing EmbeddingProvider, then register it here."
        )

    raise ValueError(
        f"Unknown embedding provider: \'{provider}\'.\\n"
        f"Supported: \'pca\', \'scgpt\', \'geneformer\', \'uce\'."
    )
'''

with open("src/embeddings/factory.py", "w", encoding="utf-8") as f:
    f.write(factory_code)

print("Updated: src/embeddings/factory.py")

Updated: src/embeddings/factory.py


In [6]:
# update_readme
with open("README.md", "r", encoding="utf-8") as f:
    readme = f.read()

old = "Currently supported: pca. Ready to extend: scgpt, geneformer, uce."

new = """Supported providers:

| Provider | Status | Notes |
|---|---|---|
| PCA | Ready | Default, no extra dependencies |
| scGPT | Implemented | Requires GPU + Linux. Windows blocked by torchtext binary incompatibility. |
| Geneformer | Interface ready | Extend src/embeddings/geneformer_provider.py |
| UCE | Interface ready | Extend src/embeddings/uce_provider.py |

To switch provider, set embeddings.provider in configs/default.yaml.
To run scGPT on a GPU machine:

    import scgpt
    scgpt.download_pretrained_model("human", save_dir="data/raw/scgpt_weights")

Then set embeddings.provider: scgpt in configs/default.yaml and run train.py."""

readme = readme.replace(old, new)

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("Updated: README.md")

Updated: README.md
